# Lab 01 · Similarity: cosine, dot, squared L2

Module B • **Core, offline.** Learn the actual arithmetic, then explain why rankings can disagree.

**Workflow:** read → predict → run → edit the marked variable → rerun → answer the linked questions in `assessments/quiz.html`.  \n**No answer needs a screenshot.** Keep your notebook output visible while taking the quiz.

# Your task board · Run first, answer later

Use the teaching cells below to determine the answers. For an **EDIT** task, change the marked variable and rerun the specified cell. The hints and full answers are intentionally **open at the end of this notebook**. Try the questions first.

**B1 · NAT · after cell 01 · B1** — For q=[1,0] and a=[2,0], what is the dot product?  
NAT: `_________`

**B2 · NAT · after cell 01 · B1** — For q=[1,0] and b=[1,1], what is the squared L2 distance?  
NAT: `_________`

**B3 · NAT · after cell 01 · B1** — What is cosine(q,a)?  
NAT: `_________`

**B4 · NAT · after cell 01 · B1** — What is cosine(q,b)? Round to 3 decimals.  
NAT: `_________`

**B5 · MCQ · after cell 01 · B2** — Which candidate wins by dot product?
  - **A.** a
  - **B.** b
  - **C.** They tie
  - **D.** Neither

**B6 · MCQ · after cell 01 · B2** — Which candidate wins by squared L2 distance?
  - **A.** a
  - **B.** b
  - **C.** They tie
  - **D.** Neither

**B7 · NAT · after cell 01 · B3** — With SCALE=3, what is dot(scaled_q,a)?  
NAT: `_________`

**B8 · NAT · after cell 01 · B3 EDIT** — After editing SCALE to 5, what is dot(scaled_q,a)?  
NAT: `_________`

**B9 · NAT · after cell 01 · B4** — For normalized q and b, squared L2 is 2−2cos. Approximate to 3 decimals.  
NAT: `_________`

**B10 · MCQ · after cell 01 · B4** — Which metric does not change when q is multiplied by a positive scalar?
  - **A.** Dot product
  - **B.** Squared L2
  - **C.** Cosine similarity
  - **D.** Vector norm

---

**Assessment:** [Interactive self-check](../assessments/quiz.html?module=B) · [Full question sheet](../assessments/QUESTION_SHEET.md). You can learn without grading; 70% unlocks the next quiz on the portal.

**Official documentation:** [scikit-learn cosine](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) · [NumPy dot](https://numpy.org/doc/stable/reference/generated/numpy.dot.html).

In [ ]:
from pathlib import Path
import numpy as np
from IPython.display import display

def find_workshop_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/embedding_workshop')]
    for p in candidates:
        if (p / 'data' / 'field_notes.md').is_file():
            return p
    raise FileNotFoundError('Cannot find data/field_notes.md. Unzip the complete workshop, not just one notebook. Check README.md.')
ROOT = find_workshop_root()
print('Workshop root:', ROOT)


## B1 · Two vectors. Three metrics.
For a query **q = [1, 0]**, and candidates **a = [2, 0]** and **b = [1, 1]**, predict the scores first.

In [ ]:
q=np.array([1.,0.]); a=np.array([2.,0.]); b=np.array([1.,1.])
def dot(u,v): return float(np.dot(u,v))
def cosine(u,v):
    den=np.linalg.norm(u)*np.linalg.norm(v)
    if den == 0: raise ValueError('cosine undefined for zero vector')
    return float(np.dot(u,v)/den)
def squared_l2(u,v): return float(np.sum((u-v)**2))
for name,v in [('a',a),('b',b)]:
    print(name, 'dot =', round(dot(q,v),4), 'cosine =', round(cosine(q,v),4), 'squared_l2 =', round(squared_l2(q,v),4))

## B2 · Rank correctly
**Higher is better** for dot and cosine similarity. **Lower is better** for distance. Distances returned by a vector database are not necessarily similarities.

In [ ]:
candidates={'a':a,'b':b}
for metric,score,descending in [('dot',dot,True),('cosine',cosine,True),('squared_l2',squared_l2,False)]:
    ranking=sorted(candidates,key=lambda k:score(q,candidates[k]),reverse=descending)
    print(f'{metric:12s} rank:', ' > '.join(ranking))

## B3 · Change only magnitude
✏️ Change **SCALE = 3** to **SCALE = 5**. Run and compare metric values. Direction remains unchanged, but magnitude-dependent scores change.

In [ ]:
SCALE = 3  # ✏️ EDIT TO 5, then restore to 3
scaled_q=q*SCALE
print('scale =', SCALE)
print('cosine(q,a) =', cosine(scaled_q,a))
print('dot(q,a) =', dot(scaled_q,a))
print('squared_l2(q,a) =', squared_l2(scaled_q,a))
print('q norm =', np.linalg.norm(scaled_q))

## B4 · Why normalized L2 and cosine can agree
For **unit vectors**, squared L2 = 2 − 2·cosine. This relationship does **not** hold if you skip normalization.

In [ ]:
def unit(u): return u/np.linalg.norm(u)
qa, aa, ba = unit(q), unit(a), unit(b)
for label,v in [('a',aa),('b',ba)]:
    print(label,'cos =', round(cosine(qa,v),6), 'squared_l2 =', round(squared_l2(qa,v),6),
          '2-2*cos =', round(2-2*cosine(qa,v),6))

## B5 · Coordinate geometry view

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,5))
for label,v,col in [('q',q,'#147d7d'),('a',a,'#d76339'),('b',b,'#6554ae')]:
    ax.arrow(0,0,*v,head_width=.065,length_includes_head=True,color=col,linewidth=2.5)
    ax.text(v[0]+.05,v[1]+.05,label,fontsize=13)
ax.axhline(0,color='#aaa');ax.axvline(0,color='#aaa')
ax.set(xlim=(-.2,2.5),ylim=(-.4,1.5),xlabel='dimension 1',ylabel='dimension 2',title='Same query, different retrieval metrics')
ax.grid(alpha=.2);ax.set_aspect('equal');plt.show()

### Finish Module B
Answer **B1–B10**. Challenge yourself to explain a different top-ranked neighbor without blaming the embedding model.

## Hints · Use these only after attempting the tasks

- **B1:** Expand q·a as a sum of coordinate-wise products.
- **B2:** Subtract b from q, square each component, and sum.
- **B3:** Parallel, positively scaled vectors have cosine 1.
- **B4:** Remember that the norm of [1,1] is √2.
- **B5:** Compute both dot products and rank descending.
- **B6:** Compute both squared distances and check for a tie.
- **B7:** Scale the query vector *before* computing the dot product.
- **B8:** Change SCALE to 5, rerun cell B3, inspect dot.
- **B9:** Use the identity for unit vectors: 2−2cos.
- **B10:** Ask which score removes the lengths of both operands.

These are learning hints, not the solutions. Scroll down for the fully open answer key.


## Open answers · Full worked key and explanations

The answers below refer to the **supplied dataset and stated default/EDIT settings**. If you changed additional inputs, your results may differ. Compare the experiment and formula—not just the number.

**B1: 2**  
1×2 + 0×0 = 2.

**B2: 1**  
(1−1)^2 + (0−1)^2 = 1.

**B3: 1**  
The vectors point in the same direction.

**B4: 0.707**  
1 / sqrt(2) ≈ .7071.

**B5: A — a**  
Dot(q,a)=2 and dot(q,b)=1.

**B6: C — They tie**  
Both a and b are at squared distance 1 from q.

**B7: 6**  
[3,0]·[2,0] = 6.

**B8: 10**  
[5,0]·[2,0] = 10.

**B9: 0.586**  
2−2×0.70710678 ≈ 0.585786.

**B10: C — Cosine similarity**  
Cosine normalizes away positive scaling.

### Extension challenge — Interview explanation

**Challenge:** Two systems use the same model and query but rank different documents. Which scoring assumptions would you inspect?

**Self-check:** Explain the trade-off aloud in 45 seconds using a concrete example.
